In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

In [25]:
# Localizar a pasta principal do projeto
root = next(
    (
        pasta for pasta in [Path.cwd(), *Path.cwd().parents]
        if (pasta / "artifacts" / "catalog_gmm.parquet").exists()
    ),
    None
)

if root is None:
    raise FileNotFoundError("Não encontrei o catálogo GMM.")

print(root)

/Users/tancrede/code/Victorpeereiraa/cross_genre_music_discovery


In [26]:
caminho_modelo = (
    root / "artifacts" / "models" / "gmm"
    / "gmm_pca3_tied15.joblib"
)

objetos = joblib.load(caminho_modelo)

gmm = objetos["model"]
pca = objetos["pca"]
scaler = objetos["scaler"]

print("Modelo carregado com sucesso!")
print("Número de clusters:", gmm.n_components)

Modelo carregado com sucesso!
Número de clusters: 15


/Users/tancrede/.pyenv/versions/3.11.18/envs/cross_genre_music_discovery/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator GaussianMixture from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/tancrede/.pyenv/versions/3.11.18/envs/cross_genre_music_discovery/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/tancrede/.pyenv/versions/3.11.18/envs/cross_genre_music_discovery/lib/python3.11/site-package

In [27]:
caminho_catalogo = root / "artifacts" / "catalog_gmm.parquet"

df = pd.read_parquet(caminho_catalogo)

# Garantir que track_id seja uma coluna
if "track_id" not in df.columns:
    df = df.reset_index()

print("Quantidade de músicas:", len(df))
print("Colunas disponíveis:")
print(df.columns.tolist())

display(df.head())

Quantidade de músicas: 80175
Colunas disponíveis:
['track_id', 'artists', 'track_name', 'genres', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'valence', 'tempo']


,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]",0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.715,87.917
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]",0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.267,77.489
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic],0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.120,76.332
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic],0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.143,181.740
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic],0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.167,119.949


In [28]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "valence",
    "tempo"
]

# Selecionar as features
X = df[audio_features]

# Aplicar o scaler treinado
X_scaled = pd.DataFrame(
    scaler.transform(X),
    columns=audio_features,
    index=df.index
)

# Aplicar o PCA treinado
X_pca = pca.transform(X_scaled)

# Identificar o cluster de cada música
df["cluster_gmm"] = gmm.predict(X_pca)

# Probabilidade de pertencer ao cluster atribuído
probabilidades = gmm.predict_proba(X_pca)

df["confianca_cluster"] = probabilidades[
    np.arange(len(df)),
    df["cluster_gmm"].to_numpy()
]

display(df.head())

/Users/tancrede/.pyenv/versions/3.11.18/envs/cross_genre_music_discovery/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(
/Users/tancrede/.pyenv/versions/3.11.18/envs/cross_genre_music_discovery/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo,cluster_gmm,confianca_cluster
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]",0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.715,87.917,11,0.412183
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]",0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.267,77.489,0,0.900407
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic],0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.120,76.332,14,0.766227
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic],0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.143,181.740,0,0.626786
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic],0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.167,119.949,14,0.549716


In [29]:
print(df["genres"].iloc[0])
print(type(df["genres"].iloc[0]))

['acoustic' 'j-pop' 'singer-songwriter' 'songwriter']
<class 'numpy.ndarray'>


In [30]:
# Criar uma cópia com os gêneros separados
df_generos = df[
    ["track_id", "cluster_gmm", "genres"]
].explode("genres").copy()

# Remover gêneros ausentes
df_generos = df_generos.dropna(subset=["genres"])

# Padronizar os nomes
df_generos["genres"] = (
    df_generos["genres"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Contar músicas de cada gênero por cluster
perfil_clusters = (
    df_generos.groupby(["cluster_gmm", "genres"])["track_id"]
    .nunique()
    .reset_index(name="quantidade")
    .sort_values(
        ["cluster_gmm", "quantidade"],
        ascending=[True, False]
    )
)

# Mostrar os 5 gêneros mais frequentes em cada cluster
top_5 = perfil_clusters.groupby("cluster_gmm").head(5)

display(top_5)

,cluster_gmm,genres,quantidade
66,0,romance,311
54,0,opera,272
34,0,guitar,210
14,0,classical,173
20,0,disney,165
...,...,...,...
1332,14,cantopop,476
1385,14,mandopop,435
1320,14,acoustic,353
1391,14,opera,333


In [31]:
# Total de músicas únicas por cluster
total_clusters = (
    df.groupby("cluster_gmm")["track_id"]
    .nunique()
)

# Adicionar o total ao perfil
perfil_clusters["total_cluster"] = (
    perfil_clusters["cluster_gmm"].map(total_clusters)
)

# Calcular a proporção do gênero no cluster
perfil_clusters["proporcao"] = (
    perfil_clusters["quantidade"]
    / perfil_clusters["total_cluster"]
)

# Ordenar pelos gêneros mais representativos
perfil_clusters = perfil_clusters.sort_values(
    ["cluster_gmm", "proporcao"],
    ascending=[True, False]
)

# Exibir os 5 principais gêneros de cada cluster
top_5 = perfil_clusters.groupby("cluster_gmm").head(5)

display(
    top_5.style.format({"proporcao": "{:.1%}"})
)

,cluster_gmm,genres,quantidade,total_cluster,proporcao
66,0,romance,311,2901,10.7%
54,0,opera,272,2901,9.4%
34,0,guitar,210,2901,7.2%
14,0,classical,173,2901,6.0%
20,0,disney,165,2901,5.7%
138,1,j-dance,434,4471,9.7%
100,1,dancehall,342,4471,7.6%
127,1,hardcore,211,4471,4.7%
116,1,funk,189,4471,4.2%
130,1,hip-hop,185,4471,4.1%


In [32]:
# Frequência de cada gênero no dataset inteiro
frequencia_global = (
    df_generos.groupby("genres")["track_id"]
    .nunique()
    / df["track_id"].nunique()
)

# Adicionar frequência global ao perfil
perfil_clusters["freq_global"] = (
    perfil_clusters["genres"].map(frequencia_global)
)

# Calcular o Lift
perfil_clusters["lift"] = (
    perfil_clusters["proporcao"]
    / perfil_clusters["freq_global"]
)

# Evitar gêneros com pouquíssimas músicas
perfil_filtrado = perfil_clusters[
    perfil_clusters["quantidade"] >= 30
].copy()

# Ordenar pelos gêneros mais característicos
perfil_filtrado = perfil_filtrado.sort_values(
    ["cluster_gmm", "lift", "quantidade"],
    ascending=[True, False, False]
)

# Mostrar os 5 principais por cluster
top_lift = (
    perfil_filtrado
    .groupby("cluster_gmm")
    .head(5)
)

display(
    top_lift[
        ["cluster_gmm", "genres", "quantidade", "proporcao", "lift"]
    ].style.format({
        "proporcao": "{:.1%}",
        "lift": "{:.2f}"
    })
)

,cluster_gmm,genres,quantidade,proporcao,lift
66,0,romance,311,10.7%,9.68
54,0,opera,272,9.4%,7.82
34,0,guitar,210,7.2%,5.89
14,0,classical,173,6.0%,5.86
46,0,jazz,89,3.1%,4.93
138,1,j-dance,434,9.7%,7.93
100,1,dancehall,342,7.6%,6.92
130,1,hip-hop,185,4.1%,4.50
116,1,funk,189,4.2%,4.20
127,1,hardcore,211,4.7%,3.88


In [33]:
from itertools import combinations, product

# Informações das músicas, sem a coluna de arrays
base = df.drop(columns=["genres"]).drop_duplicates("track_id").copy()

# Identificar artistas para evitar repetições
base["artista_id"] = (
    base["artists"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Associar cada música aos seus gêneros
candidatas = df_generos.merge(
    base,
    on=["track_id", "cluster_gmm"],
    how="inner"
)

# Guardar todos os gêneros atribuídos a cada música
generos_por_musica = (
    df_generos.groupby("track_id")["genres"]
    .agg(set)
    .to_dict()
)

# Selecionar os cinco gêneros com maior Lift por cluster
opcoes_generos = (
    perfil_filtrado
    .sort_values(
        ["cluster_gmm", "lift", "quantidade"],
        ascending=[True, False, False]
    )
    .groupby("cluster_gmm")
    .head(5)
)

display(opcoes_generos.head(10))

,cluster_gmm,genres,quantidade,total_cluster,proporcao,freq_global,lift
66,0,romance,311,2901,0.107204,0.011076,9.679182
54,0,opera,272,2901,0.093761,0.011986,7.822341
34,0,guitar,210,2901,0.072389,0.012298,5.886181
14,0,classical,173,2901,0.059635,0.010178,5.859320
46,0,jazz,89,2901,0.030679,0.006224,4.929248
138,1,j-dance,434,4471,0.097070,0.012236,7.933321
100,1,dancehall,342,4471,0.076493,0.011051,6.921922
130,1,hip-hop,185,4471,0.041378,0.009205,4.495207
116,1,funk,189,4471,0.042272,0.010053,4.204952
127,1,hardcore,211,4471,0.047193,0.012173,3.876742


In [34]:
def montar_playlist(n_clusters=1, n_musicas=5, seed=42, bloqueadas=None):

    distribuicoes = {
        1: [5],
        3: [2, 2, 1],
        5: [1, 1, 1, 1, 1]
    }

    if n_clusters not in distribuicoes or n_musicas != 5:
        raise ValueError("Use 5 músicas e 1, 3 ou 5 clusters.")

    rng = np.random.default_rng(seed)
    bloqueadas = set() if bloqueadas is None else set(bloqueadas)

    # 1. Gêneros disponíveis para cada cluster
    generos_cluster = {
        cluster: grupo["genres"].tolist()
        for cluster, grupo in opcoes_generos.groupby("cluster_gmm")
    }

    # 2. Organizar músicas por cluster e gênero
    grupos = {
        chave: grupo.sort_values(
            "confianca_cluster", ascending=False
        )
        for chave, grupo in candidatas.groupby(
            ["cluster_gmm", "genres"]
        )
    }

    # 3. Criar combinações possíveis de clusters
    clusters = sorted(generos_cluster.keys())

    combinacoes = list(combinations(clusters, n_clusters))
    rng.shuffle(combinacoes)

    def distancia_clusters(combinacao):
        if len(combinacao) == 1:
            return 0

        centros = gmm.means_[list(combinacao)]

        distancias = np.linalg.norm(
            centros[:, None, :] - centros[None, :, :],
            axis=2
        )

        return distancias[
            np.triu_indices(len(combinacao), k=1)
        ].min()

    # Para múltiplos clusters, priorizar os mais separados
    if n_clusters > 1:
        combinacoes.sort(
            key=distancia_clusters,
            reverse=True
        )

    # 4. Buscar combinação válida de clusters e gêneros
    for combinacao in combinacoes:

        opcoes = [
            generos_cluster[c] for c in combinacao
        ]

        for generos in product(*opcoes):

            # Os gêneros escolhidos precisam ser diferentes
            if len(set(generos)) != n_clusters:
                continue

            selecionadas = []
            ids_usados = bloqueadas.copy()
            artistas_usados = set()

            for cluster, genero, quantidade in zip(
                combinacao,
                generos,
                distribuicoes[n_clusters]
            ):

                grupo = grupos.get((cluster, genero))

                if grupo is None:
                    break

                # Evitar que uma música tenha outro gênero-alvo
                outros_generos = set(generos) - {genero}

                pool = grupo[
                    ~grupo["track_id"].isin(ids_usados)
                ].copy()

                pool = pool[
                    pool["track_id"].map(
                        lambda tid: generos_por_musica[tid]
                        .isdisjoint(outros_generos)
                    )
                ]

                pool = pool[
                    ~pool["artista_id"].isin(artistas_usados)
                ]

                # Priorizar músicas com confiança alta,
                # mas permitir variação entre execuções
                pool = pool.drop_duplicates("artista_id").head(30)

                if len(pool) < quantidade:
                    break

                escolhidas = pool.sample(
                    n=quantidade,
                    random_state=seed + int(cluster)
                ).copy()

                escolhidas["genero_alvo"] = genero

                selecionadas.append(escolhidas)

                ids_usados.update(escolhidas["track_id"])
                artistas_usados.update(escolhidas["artista_id"])

            # Verificar se conseguimos preencher os clusters
            if len(selecionadas) == n_clusters:

                playlist = pd.concat(
                    selecionadas, ignore_index=True
                )

                colunas = [
                    "track_id",
                    "track_name",
                    "artists",
                    "genero_alvo",
                    "cluster_gmm",
                    "confianca_cluster"
                ]

                return playlist[colunas]

    raise ValueError(
        "Não foi possível encontrar uma combinação válida."
    )

In [ ]:
playlist_1 = montar_playlist(
    n_clusters=1,
    seed=42
)

playlist_2 = montar_playlist(
    n_clusters=3,
    seed=42,
    bloqueadas=playlist_1["track_id"]
)

playlist_3 = montar_playlist(
    n_clusters=5,
    seed=42,
    bloqueadas=pd.concat([
        playlist_1["track_id"],
        playlist_2["track_id"]
    ])
)

for nome, playlist in {
    "HOMOGÊNEA": playlist_1,
    "INTERMEDIÁRIA": playlist_2,
    "ECLÉTICA": playlist_3
}.items():

    print(f"\n{'='*15} {nome} {'='*15}")

    display(playlist)

    # Save playlist for blind test
    # playlist.to_csv(f"Playlist_{nome}.csv", index=False)

    print("Clusters:", playlist["cluster_gmm"].nunique())
    print("Gêneros-alvo:", playlist["genero_alvo"].nunique())


=============== HOMOGÊNEA ===============


,track_id,track_name,artists,genero_alvo,cluster_gmm,confianca_cluster
0,12eNFSD2IVbUYNcFZXCV6x,The Green River Killer,Apartment 213,grindcore,6,0.999045
1,0EVJ6qLuqXM9ZDyBqTtYqv,Revival Spines,Trap Them,grindcore,6,0.999827
2,13wggP4SuPcy8pVVFPJMhD,Car Bombing in the Name of the Family,Deterioration,grindcore,6,0.999353
3,1qeCiaJHkMB4juwzVGOOek,Cool Guy Backed Hard,Fluoride,grindcore,6,0.999152
4,3reWy7prsDZext0iT3Tr9e,Skinjob,Hatred Surge,grindcore,6,0.999863


Clusters: 1
Gêneros-alvo: 1

=============== INTERMEDIÁRIA ===============


,track_id,track_name,artists,genero_alvo,cluster_gmm,confianca_cluster
0,5olDF40pLOsPi0JjaiY37m,De Treta Em Treta,MC Caja;Mc Gibi,funk,4,1.000000
1,24N4CgmUozgo6lwd2qM33i,Idolatria,Menor MC,funk,4,1.000000
2,29cWhgW69qlqEcji31Dr0D,Edge Of Dawn,Gary Stadler;Stephannie,new-age,7,0.999859
3,5Qj0IHHYHoHaiAc6VXK5hf,Hands Of Silk,Michael Whalen,new-age,7,0.999915
4,0fmwyv8VP8Jh71e87oCHv1,Through Struggle,As I Lay Dying,metalcore,13,0.884795


Clusters: 3
Gêneros-alvo: 3

=============== ECLÉTICA ===============


,track_id,track_name,artists,genero_alvo,cluster_gmm,confianca_cluster
0,79dPOFmCNvwKAcDur9ZP0v,Aalystice,Ben Lukas Boysen;Sebastian Plano,idm,2,0.901859
1,3N6isgIE6SAFbRfRIbQ3sd,Morena Cor do Pecado,MC Kevin o Chris;Dj Juninho 22,funk,4,1.000000
2,3lGmH2vFtfaTQJ5Bq7tIMZ,First Impression,Floyd Cramer,honky-tonk,5,0.827442
3,5ByhNiOjVNhY2XNzt6I6pd,Alpha Legion,Imperatorz;Disarray,hardstyle,12,0.999123
4,19JvAP6RDVLg1xJBeRLzje,Holdfast,The Devil Wears Prada,metalcore,13,0.883670


Clusters: 5
Gêneros-alvo: 5
